# Cell 1 — Config

In [1]:
import os, glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision import transforms, models
from PIL import Image
import random

device = 'cuda' if torch.cuda.is_available() else 'cpu'

DATA_ROOT   = "/kaggle/input/datasets/phongkaggle/shelf-management-product/SHelf mAnagement Product"  # SỬA LẠI
TRAIN_DIR   = os.path.join(DATA_ROOT, "training_set")
TEST_DIR    = os.path.join(DATA_ROOT, "test_set")

IMG_SIZE     = 224
EMBED_DIM    = 256
P_CLASSES    = 16     # số SKU khác nhau / batch
K_IMAGES     = 4      # số ảnh / SKU / batch  -> batch_size = P*K = 64
EPOCHS       = 30
LR           = 1e-4
MARGIN       = 1.0    # giống paper: margin=1.0

# Cell 2 — Quét cấu trúc folder

In [2]:
def scan_dataset(root_dir):
    samples = []               # (path, label_str)
    for category in sorted(os.listdir(root_dir)):
        cat_path = os.path.join(root_dir, category)
        if not os.path.isdir(cat_path):
            continue
        for sku in sorted(os.listdir(cat_path)):
            sku_path = os.path.join(cat_path, sku)
            if not os.path.isdir(sku_path):
                continue
            label = f"{category}_{sku}"   # ghép category+sku để đảm bảo unique tuyệt đối
            for img_path in glob.glob(os.path.join(sku_path, "*.jpg")):
                samples.append((img_path, label))
    return samples

train_samples = scan_dataset(TRAIN_DIR)
test_samples  = scan_dataset(TEST_DIR)

all_labels = sorted(set(s[1] for s in train_samples))
label_to_idx = {lb: i for i, lb in enumerate(all_labels)}
print(f"train images={len(train_samples)} | SKUs={len(all_labels)} | test images={len(test_samples)}")

train images=45277 | SKUs=15847 | test images=813


# Cell 3 — Dataset + PK Sampler (bắt buộc cho batch-hard triplet)

In [3]:
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class SKUDataset(Dataset):
    def __init__(self, samples, label_to_idx, transform):
        self.samples = samples
        self.label_to_idx = label_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        img = self.transform(img)
        return img, self.label_to_idx[label]

class PKSampler(Sampler):
    """Mỗi batch: chọn ngẫu nhiên P class (SKU), mỗi class lấy K ảnh (lặp lại nếu không đủ K)."""
    def __init__(self, samples, label_to_idx, p=P_CLASSES, k=K_IMAGES, iterations=None):
        self.p, self.k = p, k
        self.label_to_indices = {}
        for i, (_, label) in enumerate(samples):
            self.label_to_indices.setdefault(label_to_idx[label], []).append(i)
        self.labels = list(self.label_to_indices.keys())
        self.iterations = iterations or (len(samples) // (p * k))

    def __iter__(self):
        for _ in range(self.iterations):
            batch = []
            chosen_labels = random.sample(self.labels, self.p)
            for lb in chosen_labels:
                idxs = self.label_to_indices[lb]
                replace = len(idxs) < self.k
                batch.extend(np.random.choice(idxs, self.k, replace=replace).tolist())
            yield batch

    def __len__(self):
        return self.iterations

train_ds = SKUDataset(train_samples, label_to_idx, train_tf)
sampler = PKSampler(train_samples, label_to_idx, p=P_CLASSES, k=K_IMAGES)
train_loader = DataLoader(train_ds, batch_sampler=sampler, num_workers=2)

# Cell 4 — Model: MobileNetV3-Large → embedding 256-d, L2-normalize

In [4]:
class EmbeddingNet(nn.Module):
    def __init__(self, embed_dim=EMBED_DIM, pretrained=True):
        super().__init__()
        backbone = models.mobilenet_v3_large(weights="DEFAULT" if pretrained else None)
        in_features = backbone.classifier[0].in_features
        backbone.classifier = nn.Identity()   # bỏ phần classifier gốc
        self.backbone = backbone
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(in_features, embed_dim)

    def forward(self, x):
        feat = self.backbone(x)
        feat = self.dropout(feat)
        emb = self.fc(feat)
        return F.normalize(emb, p=2, dim=1)   # L2-normalize như paper

model = EmbeddingNet().to(device)

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 73.9MB/s]


# Cell 5 — Batch-Hard Triplet Loss (online hard mining, Hermans et al. 2017)

In [5]:
def batch_hard_triplet_loss(embeddings, labels, margin=MARGIN):
    dist = torch.cdist(embeddings, embeddings, p=2)          # (B,B) ma trận khoảng cách Euclid
    labels = labels.unsqueeze(1)
    same_mask = (labels == labels.t())
    diff_mask = ~same_mask

    # hardest positive: xa nhất trong cùng class (bỏ qua chính nó = 0 trên đường chéo)
    dist_pos = dist.clone()
    dist_pos[~same_mask] = -1
    hardest_pos = dist_pos.max(dim=1).values

    # hardest negative: gần nhất khác class
    dist_neg = dist.clone()
    dist_neg[~diff_mask] = float('inf')
    hardest_neg = dist_neg.min(dim=1).values

    loss = F.relu(hardest_pos - hardest_neg + margin)
    return loss.mean()

# Cell 6 — Train

In [6]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        embeddings = model(imgs)
        loss = batch_hard_triplet_loss(embeddings, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Epoch {epoch+1}/{EPOCHS} | triplet_loss={total_loss/len(train_loader):.4f}")

torch.save(model.state_dict(), "embedding_mobilenetv3.pth")

Epoch 1/30 | triplet_loss=0.5185
Epoch 2/30 | triplet_loss=0.3438
Epoch 3/30 | triplet_loss=0.2767
Epoch 4/30 | triplet_loss=0.2389
Epoch 5/30 | triplet_loss=0.2139
Epoch 6/30 | triplet_loss=0.2021
Epoch 7/30 | triplet_loss=0.1859
Epoch 8/30 | triplet_loss=0.1731
Epoch 9/30 | triplet_loss=0.1618
Epoch 10/30 | triplet_loss=0.1586
Epoch 11/30 | triplet_loss=0.1507
Epoch 12/30 | triplet_loss=0.1444
Epoch 13/30 | triplet_loss=0.1359
Epoch 14/30 | triplet_loss=0.1368
Epoch 15/30 | triplet_loss=0.1316
Epoch 16/30 | triplet_loss=0.1275
Epoch 17/30 | triplet_loss=0.1255
Epoch 18/30 | triplet_loss=0.1210
Epoch 19/30 | triplet_loss=0.1207
Epoch 20/30 | triplet_loss=0.1162
Epoch 21/30 | triplet_loss=0.1145
Epoch 22/30 | triplet_loss=0.1165
Epoch 23/30 | triplet_loss=0.1113
Epoch 24/30 | triplet_loss=0.1101
Epoch 25/30 | triplet_loss=0.1096
Epoch 26/30 | triplet_loss=0.1060
Epoch 27/30 | triplet_loss=0.1086
Epoch 28/30 | triplet_loss=0.1129
Epoch 29/30 | triplet_loss=0.1092
Epoch 30/30 | triplet_l

# Cell 7 — Trích xuất embedding cho toàn bộ gallery (train) + test, lưu file để dùng FAISS local

In [7]:
@torch.no_grad()
def extract_embeddings(samples, model, transform, batch_size=64):
    model.eval()
    ds = SKUDataset(samples, label_to_idx, transform)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2)

    all_emb, all_labels = [], []
    idx = 0
    for imgs, labels in loader:
        emb = model(imgs.to(device)).cpu().numpy()
        all_emb.append(emb)
        all_labels.append(labels.numpy())
    return np.concatenate(all_emb), np.concatenate(all_labels)

gallery_emb, gallery_labels = extract_embeddings(train_samples, model, eval_tf)
test_emb, test_labels       = extract_embeddings(
    [(p, l) for p, l in test_samples if l in label_to_idx],  # bỏ SKU test không có trong train (nếu có)
    model, eval_tf
)

idx_to_label = {v: k for k, v in label_to_idx.items()}
image_paths_gallery = [s[0] for s in train_samples]

np.savez(
    "gallery_embeddings.npz",
    embeddings=gallery_emb.astype(np.float32),
    labels=gallery_labels,
    label_names=np.array([idx_to_label[i] for i in gallery_labels]),
    image_paths=np.array(image_paths_gallery),
)
np.savez(
    "test_embeddings.npz",
    embeddings=test_emb.astype(np.float32),
    labels=test_labels,
    label_names=np.array([idx_to_label[i] for i in test_labels]),
)
print("Đã lưu gallery_embeddings.npz và test_embeddings.npz")

Đã lưu gallery_embeddings.npz và test_embeddings.npz
